In [1]:
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression


# ---------------------------------------------------
# Gradient Descent (stable + precise + overflow safe)
# ---------------------------------------------------
def gradient_descent(x, y, learning_rate, max_iterations=1_000_000):
    m = 0.0
    b = 0.0
    n = len(x)
    prev_cost = float("inf")

    for i in range(max_iterations):
        y_pred = m * x + b
        error = y - y_pred

        # Cost
        cost = (1/n) * np.sum(error ** 2)

        # Stop when cost stops improving
        if math.isclose(cost, prev_cost, rel_tol=1e-20):
            break

        # Gradients
        md = -(2/n) * np.sum(x * error)
        bd = -(2/n) * np.sum(error)

        # Update
        m_new = m - learning_rate * md
        b_new = b - learning_rate * bd

        # Overflow protection
        if not np.isfinite(m_new) or not np.isfinite(b_new):
            return float("inf"), float("inf"), float("inf")

        m, b = m_new, b_new
        prev_cost = cost

    return m, b, cost


# ---------------------------------------------------
# Tune Learning Rate (increase until cost stops improving)
# ---------------------------------------------------
def tune_learning_rate(x, y, start_lr, lr_step, fixed_iterations):
    current_lr = start_lr
    best_lr = start_lr
    best_cost = float("inf")

    while True:
        m, b, cost = gradient_descent(x, y, current_lr, fixed_iterations)
        print(f"Trying learning_rate={current_lr:.6f}, cost={cost:.10f}")

        if (not np.isfinite(cost)) or math.isclose(cost, best_cost, rel_tol=1e-12) or cost > best_cost:
            print(f"Cost increased at learning_rate={current_lr:.6f}. Reverting to {best_lr:.6f}")
            break

        best_cost = cost
        best_lr = current_lr
        current_lr += lr_step

    return best_lr, best_cost


# ---------------------------------------------------
# Tune Iterations (increase until cost stops improving)
# ---------------------------------------------------
def tune_iterations(x, y, learning_rate, start_iterations, iter_step):
    current_iterations = start_iterations
    best_iterations = start_iterations
    best_cost = float("inf")

    while True:
        m, b, cost = gradient_descent(x, y, learning_rate, current_iterations)
        print(f"Trying iterations={current_iterations}, cost={cost:.10f}")

        if (not np.isfinite(cost)) or math.isclose(cost, best_cost, rel_tol=1e-12) or cost > best_cost:
            print(f"Cost increased at iterations={current_iterations}. Reverting to {best_iterations}")
            break

        best_cost = cost
        best_iterations = current_iterations
        current_iterations += iter_step

    return best_iterations, best_cost


# ---------------------------------------------------
# MAIN SCRIPT
# ---------------------------------------------------
if __name__ == "__main__":

    # Load CSV
    df = pd.read_csv("test_scores.csv")
    x = df["math"].values.astype(float)
    y = df["cs"].values.astype(float)

    # Step 1: Tune learning rate
    best_lr, _ = tune_learning_rate(
        x, y,
        start_lr=0.0001,
        lr_step=0.0001,
        fixed_iterations=500
    )

    # Step 2: Tune iterations
    best_iters, _ = tune_iterations(
        x, y,
        learning_rate=best_lr,
        start_iterations=200,
        iter_step=200
    )

    # Step 3: Final gradient descent
    m, b, cost = gradient_descent(x, y, best_lr, best_iters)

    print("\nFinal selected values:")
    print("learning_rate =", best_lr)
    print("iterations    =", best_iters)
    print("m             =", m)
    print("b             =", b)
    print("final cost    =", cost)

    # Step 4: Compare with sklearn
    model = LinearRegression()
    model.fit(df[['math']], df['cs'])

    print("\nSklearn Results:")
    print("m =", model.coef_[0])
    print("b =", model.intercept_)


Trying learning_rate=0.000100, cost=31.8094899192
Trying learning_rate=0.000200, cost=31.8071550528
Trying learning_rate=0.000300, cost=16936984997384114228598983343771234941551131327708581622565766875681977985239570850063174171994350047622132018014142972214286047336244957393477756894482521544438682882721725290382623246786508563906081764739060281320522369680160206593625280536169368454427100756687454208.0000000000
Cost increased at learning_rate=0.000300. Reverting to 0.000200
Trying iterations=200, cost=31.8099657548
Trying iterations=400, cost=31.8080876549
Trying iterations=600, cost=31.8062267231
Trying iterations=800, cost=31.8043828025
Trying iterations=1000, cost=31.8025557375
Trying iterations=1200, cost=31.8007453740
Trying iterations=1400, cost=31.7989515595
Trying iterations=1600, cost=31.7971741426
Trying iterations=1800, cost=31.7954129734
Trying iterations=2000, cost=31.7936679034
Trying iterations=2200, cost=31.7919387854
Trying iterations=2400, cost=31.7902254736
Trying 

KeyboardInterrupt: 